# Data Pipeline: Raw Tables -> Vendor Summary Table

**Purpose of this notebook:** show *how* `data/vendor_sales_summary.csv` (used by `02_vendor_performance_analysis.ipynb`) gets built from the original source tables.

> **Note:** the raw source tables (`purchases`, `sales`, `purchase_prices`, `vendor_invoice`) are large operational exports and aren't included in this repo — only the final aggregated `vendor_sales_summary.csv` is. This notebook is kept here as **documentation of the pipeline logic**, and will run end-to-end if you have those raw CSVs and point `ingestion_db.py` at them.

**Pipeline overview:**
1. `scripts/ingestion_db.py` loads each raw CSV into its own table in a local SQLite database (`inventory.db`).
2. This notebook explores those raw tables to decide what needs to be joined.
3. `scripts/get_vendor_summary.py` (same SQL as below) builds the final `vendor_sales_summary` table and exports it to CSV — that CSV is the starting point for the actual analysis notebook.


## 1. Setup

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("inventory.db")


## 2. What tables exist, and what's in them?

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables


In [ ]:
for table in tables["name"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS count FROM {table}", conn)["count"].values[0]
    print(f"{table}: {count:,} rows")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 3", conn))


**Decision:** `begin_inventory` and `end_inventory` capture stock levels at the start/end of the year — useful for a stock-count reconciliation, but not needed for vendor/brand profitability. They're excluded from the summary table.

The four tables that matter:
- `purchases` — what was bought, from whom, how much, at what price
- `purchase_prices` — reference pricing/volume per brand
- `sales` — what was sold, and for how much
- `vendor_invoice` — freight cost per vendor invoice


## 3. Sanity-check one vendor across tables

In [ ]:
VENDOR = 4466  # arbitrary vendor used to eyeball the join logic

purchases = pd.read_sql_query(f"SELECT * FROM purchases WHERE VendorNumber = {VENDOR}", conn)
sales = pd.read_sql_query(f"SELECT * FROM sales WHERE VendorNo = {VENDOR}", conn)

print("Purchases:", purchases.shape)
print("Sales:", sales.shape)


In [ ]:
# Purchases, grouped the same way the final summary will group them
purchases.groupby(["Brand", "PurchasePrice"])[["Quantity", "Dollars"]].sum()


In [ ]:
# Sales, grouped by brand
sales.groupby("Brand")[["SalesQuantity", "SalesDollars", "SalesPrice", "ExciseTax"]].sum()


**Takeaway:** `purchases` and `sales` both key on `Brand` + vendor, so they can be joined on `(VendorNumber, Brand)`. `vendor_invoice` only has freight totals per vendor (not per brand), so it's joined at the vendor level only.


## 4. Build the summary table

One query, three CTEs (freight, purchases, sales), joined together. This is the same SQL used in `scripts/get_vendor_summary.py` — see that file for the full, reusable version.

In [ ]:
vendor_sales_summary = pd.read_sql_query("""
WITH FreightSummary AS (
    SELECT VendorNumber, SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),
PurchaseSummary AS (
    SELECT
        p.VendorNumber, p.VendorName, p.Brand, p.Description,
        p.PurchasePrice, pp.Price AS ActualPrice, pp.Volume,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars) AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
),
SalesSummary AS (
    SELECT VendorNo, Brand,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)
SELECT
    ps.VendorNumber, ps.VendorName, ps.Brand, ps.Description, ps.PurchasePrice,
    ps.ActualPrice, ps.Volume, ps.TotalPurchaseQuantity, ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity, ss.TotalSalesDollars, ss.TotalSalesPrice, ss.TotalExciseTax,
    fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss ON ps.VendorNumber = ss.VendorNo AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs ON ps.VendorNumber = fs.VendorNumber
ORDER BY ps.TotalPurchaseDollars DESC
""", conn)

vendor_sales_summary.shape


**Why pre-aggregate instead of joining on the fly every time?** This join touches every row of `purchases` and `sales` — expensive to repeat. Running it once and saving the result means every downstream analysis (and the Power BI dashboard) just reads a small, ready-made table instead of re-running a heavy join.

## 5. Clean the result

In [ ]:
vendor_sales_summary.dtypes


In [ ]:
vendor_sales_summary.isnull().sum()


**Issues found:**
- `Volume` is stored as text, not a number.
- Some rows have missing sales figures — products that were purchased but never sold.
- Vendor/product names have stray whitespace.


In [ ]:
vendor_sales_summary["Volume"] = vendor_sales_summary["Volume"].astype("float")
vendor_sales_summary.fillna(0, inplace=True)
vendor_sales_summary["VendorName"] = vendor_sales_summary["VendorName"].str.strip()
vendor_sales_summary["Description"] = vendor_sales_summary["Description"].str.strip()

# Business metrics used throughout the analysis notebook
vendor_sales_summary["GrossProfit"] = vendor_sales_summary["TotalSalesDollars"] - vendor_sales_summary["TotalPurchaseDollars"]
vendor_sales_summary["ProfitMargin"] = (vendor_sales_summary["GrossProfit"] / vendor_sales_summary["TotalSalesDollars"]) * 100
vendor_sales_summary["StockTurnover"] = vendor_sales_summary["TotalSalesQuantity"] / vendor_sales_summary["TotalPurchaseQuantity"]
vendor_sales_summary["SalesToPurchaseRatio"] = vendor_sales_summary["TotalSalesDollars"] / vendor_sales_summary["TotalPurchaseDollars"]


## 6. Save it

In [ ]:
vendor_sales_summary.to_sql("vendor_sales_summary", conn, if_exists="replace", index=False)
vendor_sales_summary.to_csv("../data/vendor_sales_summary.csv", index=False)
print("Saved", vendor_sales_summary.shape, "rows/cols")


---
**Next:** open `02_vendor_performance_analysis.ipynb` — it picks up from `data/vendor_sales_summary.csv` and answers the actual business questions.